# BindCraft (`FreeBindCraft`) *De Novo* Protein Binder Design with `foldrun-shim`

This notebook adapts the official [**`martinpacesa/BindCraft`**](https://github.com/martinpacesa/BindCraft) Colab notebook ([`notebooks/BindCraft.ipynb`](https://github.com/martinpacesa/BindCraft/blob/main/notebooks/BindCraft.ipynb), [`settings_target/PDL1.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_target/PDL1.json), [`settings_advanced/default_4stage_multimer.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_advanced/default_4stage_multimer.json), and [`settings_filters/default_filters.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_filters/default_filters.json)) and **FreeBindCraft (`--no-pyrosetta`)** to execute on **FoldRun**'s auto-scaling Google Cloud Vertex AI A100 GPU Pipelines via **`foldrun-shim`**.

BindCraft designs *de novo* protein binders against arbitrary target structures and hotspot residues by backpropagating directly through **AlphaFold 2 Multimer / PTM** networks (4-stage ColabDesign hallucination: continuous logits $\to$ softmax $\to$ Gumbel-softmax steerable one-hot $\to$ greedy mutation), redesigning the non-interface scaffold core with **SolubleMPNN**, and validating candidate complexes with AlphaFold 2 + biophysical interface scoring.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Notebook**:
  - Official BindCraft Colab Notebook: [`martinpacesa/BindCraft` `notebooks/BindCraft.ipynb`](https://github.com/martinpacesa/BindCraft/blob/main/notebooks/BindCraft.ipynb) ([Open Original in Colab](https://colab.research.google.com/github/martinpacesa/BindCraft/blob/main/notebooks/BindCraft.ipynb))
  - Upstream Target & Filter Settings: [`settings_target/PDL1.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_target/PDL1.json), [`settings_advanced/default_4stage_multimer.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_advanced/default_4stage_multimer.json), [`settings_filters/default_filters.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_filters/default_filters.json)
- **Upstream Repositories**: [`martinpacesa/BindCraft`](https://github.com/martinpacesa/BindCraft) & [`sokrypton/ColabDesign`](https://github.com/sokrypton/ColabDesign) (plus PyRosetta-free `FreeBindCraft` open-source scoring via `OpenMM`, `FASPR`, `FreeSASA`, and `sc-rs`)
- **Authors & Credits**: Martin Pacesa, Lennart Nickel, Christian Schellhaas, Joseph Schmidt, Ekaterina Pyatova, Lucas Kissling, Patrick Barendse, Jagrity Choudhury, Srajan Kapoor, Ana Alcaraz-Serna, Yehlin Cho, Kourosh H. Ghamary, Laura Vinué, Brahm J. Yachnin, Andrew M. Wollacott, Stephen Buckley, Sandrine Georgeon, Casper A. Goverde, Georgios N. Hatzopoulos, Pierre Gönczy, Y. D. Muller, G. Schwank, D. C. Swarts, Sergey Ovchinnikov, and Bruno E. Correia (**Laboratory of Protein Design and Immunoengineering (LPDI), EPFL & MIT / Harvard ColabDesign**).
- **License**: MIT License.
- **Primary Citation**:
  1. Pacesa, M., Nickel, L., Schellhaas, C. *et al.* "BindCraft: one-shot design of functional protein binders." *Nature* (2025) / *bioRxiv* (2024). [doi:10.1101/2024.09.30.615802](https://doi.org/10.1101/2024.09.30.615802)

---

## Why Run BindCraft Through `foldrun-shim` Instead of the Public `BindCraft.ipynb` Colab?

| Capability | Public `BindCraft.ipynb` Colab Notebook | FoldRun Cloud Execution via `foldrun-shim` |
| :--- | :--- | :--- |
| **PyRosetta Commercial License** | Installs proprietary `pyrosetta` wheels from `west.rosettacommons.org` and runs `pr.init(...)` + `pr_relax(...)`, requiring a commercial license for industry/biotech use. | Runs **`FreeBindCraft --no-pyrosetta`** out of the box, replacing PyRosetta with open-source **`OpenMM`** relaxation, **`FASPR`** side-chain repacking, **`FreeSASA`** solvent-accessible surface area ($\Delta\text{SASA}$), and **`sc-rs`** shape complementarity alongside pre-compiled `DAlphaBall.gcc` and `dssp`. |
| **5.3 GB AlphaFold 2 + SolubleMPNN Weights** | Downloads `alphafold_params_2022-12-06.tar` (5.3 GB) via `aria2c` and clones `sokrypton/ColabDesign` into `/content/bindcraft/params` on every fresh Colab session. | Zero weight downloads: reuses **shared AlphaFold 2 weights** at `/mnt/nfs/foldrun/alphafold2/params` (`models: [af2, bindcraft]` in `databases.yaml`, shared with AlphaFold 2 with zero duplication!) and **SolubleMPNN weights** at `/mnt/nfs/foldrun/bindcraft/mpnn_weights` (`bindcraft_mpnn_params`), symlinked into `/opt/bindcraft/params` and `/opt/bindcraft/mpnn_weights` at container startup. |
| **4-Stage AF2 Backpropagation & Multi-GPU Scaling** | Runs trajectories sequentially in an infinite `while True:` loop on a single Colab GPU. Backpropagating through AF2 multimer networks is slow and memory-intensive on a single T4 GPU (often taking hours per binder and timing out). | Uses a 2-stage Vertex AI Pipeline (`ConfigureSeeds -> dsl.ParallelFor[PredictOp]`) to fan out `--num_model_seeds` (`1..50`) independent hallucination trajectories **in parallel** across dedicated **NVIDIA A100 40 GB** (`a2-highgpu-1g`, $\le 450$ total residues) or **NVIDIA A100 80 GB** (`a2-ultragpu-1g`, $> 450$ total residues) GPUs with DWS `FLEX_START`. |
| **Built-In Runtime & CUDA Safeguards** | Breaks when newer `colabdesign` releases drop the `weights` argument from `mk_mpnn_model.__init__`, or when `OpenMM`'s CUDA compiler fails with `CUDA_ERROR_UNSUPPORTED_PTX_VERSION (222)` on cloud GPU drivers. | Automatically injects a `sitecustomize.py` compatibility shim for `colabdesign.mpnn.model.mk_mpnn_model.__init__(..., weights='soluble')` and routes `OpenMM` relaxation onto the `CPU` platform (`OPENMM_DEFAULT_PLATFORM=CPU`) while keeping JAX/AF2 hallucination on the A100 GPU. |
| **Target Structure Input (`.pdb` / `.cif` / RCSB ID)** | Requires manually uploading a trimmed `.pdb` file to `/content/bindcraft/example/PDL1.pdb`. | Supports **both** local `.pdb`/`.cif` files in `starting_pdb` (automatically bundled into `assets.tar.gz` by `foldrun-shim`) **and** 4-character RCSB PDB IDs (`"starting_pdb": "4Z18"`, fetched server-side by `target_resolver.py` and converted from `.cif` to `.pdb` via `gemmi`/BioPython). |
| **Disconnect Resilience (`.foldrun_job.json`)** | Requires mounting Google Drive (`google.colab.drive.mount`) and loses in-flight trajectory backpropagation if the browser tab disconnects. | Persists `<out_dir>/.foldrun_job.json` upon `202 Accepted` submission and deduplicates jobs via `input_sha256`. Re-running an interrupted cell re-attaches to the running Vertex AI `PipelineJob` and materializes `<out_dir>/<stem>/Accepted/final_design_stats.csv` and `Accepted/ranked_0.cif`. |


## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
import json
import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")
os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")
# Optional: set FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT if your user account delegates to an invoker SA
# os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = "foldrun-invoker@your-project.iam.gserviceaccount.com"

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ["FOLDRUN_GCS_BUCKET"])
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pandas",
        "pyyaml",
    ],
    check=False,
)

import foldrun_shim
from foldrun_shim import bindcraft
from foldrun_shim.bundler import bundle_local_inputs

print(f"foldrun-shim version : {getattr(foldrun_shim, '__version__', 'unknown')}")
print(f"bindcraft CLI in PATH: {shutil.which('bindcraft') or 'available via foldrun_shim.cli'}")
print(f"FOLDRUN_GATEWAY_URL  : {os.environ['FOLDRUN_GATEWAY_URL']}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: "auto" (default), "L4" (fast & cost-effective), "A100" (40GB), or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "L4"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original Upstream Example (Unmodified / Verbatim Workflow)

### Original `martinpacesa/BindCraft` Upstream `PDL1.json` Example & CLI

In the upstream [`martinpacesa/BindCraft` repository](https://github.com/martinpacesa/BindCraft), binder design campaigns are executed via the command line:
```bash
!bindcraft --settings './settings_target/PDL1.json' --filters './settings_filters/default_filters.json' --advanced './settings_advanced/default_4stage_multimer.json'
```
Standard BindCraft requires local PyRosetta licenses, local AlphaFold 2 model parameters, and local SolubleMPNN weights.


In [ ]:
import urllib.request
from pathlib import Path

# Ensure target PDB exists (download 4Z18 from RCSB if missing or truncated)
pdb_path = Path("target_pdl1.pdb")
if not pdb_path.exists() or pdb_path.stat().st_size < 10000:
    try:
        urllib.request.urlretrieve("https://files.rcsb.org/download/4Z18.pdb", str(pdb_path))
        print(f"Downloaded 4Z18.pdb to {pdb_path} ({pdb_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"Failed to download 4Z18.pdb ({e})")
else:
    print(f"Using existing {pdb_path} ({pdb_path.stat().st_size} bytes)")

# Write upstream-style settings_target/PDL1.json, settings_filters/default_filters.json,
# and settings_advanced/default_4stage_multimer.json:
target_dir = Path("./settings_target")
filters_dir = Path("./settings_filters")
advanced_dir = Path("./settings_advanced")
for d in (target_dir, filters_dir, advanced_dir):
    d.mkdir(parents=True, exist_ok=True)

pdl1_upstream_settings = {
    "design_path": "./results/pdl1_binder",
    "binder_name": "PDL1_Binder",
    "starting_pdb": "./target_pdl1.pdb",
    "chains": "A",
    "target_hotspot_residues": "54,56,66",
    "lengths": [80, 120],
    "number_of_final_designs": 2,
}
(target_dir / "PDL1.json").write_text(
    json.dumps(pdl1_upstream_settings, indent=2), encoding="utf-8"
)

(filters_dir / "default_filters.json").write_text(
    json.dumps(
        {
            "Average_pLDDT": {"threshold": 0.8, "higher": True},
            "Average_i_pTM": {"threshold": 0.5, "higher": True},
        },
        indent=2,
    ),
    encoding="utf-8",
)
(advanced_dir / "default_4stage_multimer.json").write_text(
    json.dumps(
        {
            "design_algorithm": "4stage",
            "use_multimer_design": True,
            "soft_iterations": 75,
            "temporary_iterations": 45,
            "hard_iterations": 5,
            "greedy_iterations": 15,
            "enable_mpnn": True,
            "mpnn_weights": "soluble",
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(
    "Wrote settings_target/PDL1.json (design_path='./results/pdl1_binder'), settings_filters/default_filters.json, and settings_advanced/default_4stage_multimer.json"
)

In [ ]:
# Verbatim upstream BindCraft CLI invocation:
# foldrun-shim intercepts this command, bundles target_pdl1.pdb plus --filters and --advanced JSONs into assets.tar.gz,
# automatically reads "design_path": "./results/pdl1_binder" from PDL1.json when --out_dir is omitted,
# and runs FreeBindCraft on Vertex AI A100 GPUs.
!bindcraft --settings './settings_target/PDL1.json' --filters './settings_filters/default_filters.json' --advanced './settings_advanced/default_4stage_multimer.json'

---
## Part B — FoldRun Additions (`foldrun-shim` Python API, `/mnt/nfs/foldrun` Shared Databases & Weights, Local Asset Bundling, and Multi-GPU Seed Parallelism)

FoldRun transforms BindCraft into an enterprise cloud service:
1. **Server-Side RCSB PDB Resolution (`starting_pdb: "4Z18"`)**: Resolves directly from `/mnt/nfs/foldrun/pdb_mmcif` without requiring local structure downloads.
2. **Pre-Mounted Parameters on NFS**: Mounts `/mnt/nfs/foldrun/alphafold2/params` and `/mnt/nfs/foldrun/bindcraft/mpnn_weights` across all GPU workers.
3. **Multi-GPU Parallel Fan-Out (`--num_concurrent_tasks=4` / `--num_model_seeds=5`)**: Distributes independent binder design trajectories in parallel across dedicated NVIDIA A100 GPUs.
4. **Python Notebook Helper API (`bindcraft.predict`)**: Clean programmatic interface with status cards and downstream biophysical analysis.


## 2. How FoldRun's Cloud Filestore NFS Database & Shared Weights Are Leveraged

Unlike forward folding models that spend time in CPU MSA search, BindCraft hallucinates *de novo* binders by backpropagating gradients directly through **AlphaFold 2** weights (`mk_afdesign_model(protocol="binder")`) and optimizing scaffold sequences with **SolubleMPNN** (`mk_mpnn_model(..., weights="soluble")`).

In FoldRun (`foldrun-agent/foldrun_app/databases.yaml` and `foldrun_app/models/bindcraft/pipeline/components/predict.py`), every BindCraft A100 GPU worker mounts the shared **2.5 TB `BASIC_SSD` Cloud Filestore NFS volume** at `/mnt/nfs/foldrun`:

### 2.1 Shared AlphaFold 2 Weights + SolubleMPNN Weights on `/mnt/nfs/foldrun`

| `databases.yaml` Entry | `models` Tag | NFS Path (`/mnt/nfs/foldrun/...`) | Container Runtime Symlink (`predict_bindcraft`) | Purpose in BindCraft |
| :--- | :--- | :--- | :--- | :--- |
| **`alphafold_params`** | `[af2, bindcraft]` | `/mnt/nfs/foldrun/alphafold2/params` | `/opt/bindcraft/params/*` (`advanced_data["af_params_dir"] = "/opt/bindcraft"`) | Provides `params_model_{1..5}_ptm.npz` and `params_model_{1..5}_multimer_v3.npz` (~5.3 GB) for 4-stage ColabDesign hallucination, complex re-prediction (`predict_binder_complex`), and monomer validation (`predict_binder_alone`). **Downloaded once and shared with AlphaFold 2 with zero duplication!** |
| **`bindcraft_mpnn_params`** | `[bindcraft]` | `/mnt/nfs/foldrun/bindcraft/mpnn_weights` | `/opt/bindcraft/mpnn_weights/*` | Provides SolubleMPNN and ProteinMPNN checkpoints (`v_48_002.pt`, `v_48_010.pt`, `v_48_020.pt`, `v_48_030.pt`, `soluble_model_weights/*`) from `sokrypton/ColabDesign` for stage-4 sequence redesign (`mpnn_gen_sequence`). |
| **Container Biophysical Tools** | Built into image | N/A (pre-compiled in `/opt/bindcraft/functions/`) | `dssp`, `DAlphaBall.gcc`, `faspr` | Configured automatically in `advanced_settings.json` (`dssp_path`, `dalphaball_path`, `faspr_path`) alongside `OpenMM`, `FreeSASA`, and `sc-rs` for `--no-pyrosetta` relaxation and interface scoring. |

### 2.2 Two Target Structure Resolution Modes (`starting_pdb`)
BindCraft settings files (`pdl1_settings.json`) specify a target structure via `starting_pdb` (or FoldRun's aliases `target_pdb` / `target_structure`). FoldRun supports two complementary resolution modes:
- **Mode A — Local `.pdb` / `.cif` Asset Bundling (`"starting_pdb": "./target_pdl1.pdb"`)**:
  1. `foldrun_shim.bundler._collect_bindcraft_assets` inspects `starting_pdb`, `target_pdb`, and `target_structure` in your JSON/YAML settings file.
  2. If the value is a relative local file path (not a 4-character PDB ID or `gs://` URI), `foldrun-shim` verifies path containment inside the input directory and packages the structure into `assets.tar.gz`.
  3. `foldrun-gateway` (`staging.py`) extracts `assets.tar.gz` to `gs://<bucket>/gateway_jobs/<job_id>/inputs/assets/` and rewrites `starting_pdb` to the staged `gs://` URI.
- **Mode B — Server-Side 4-Character RCSB PDB Resolution (`"starting_pdb": "4Z18"`)**:
  1. When `starting_pdb` is a 4-character RCSB PDB ID (such as `"4Z18"`), `foldrun-shim` does not require any local `.pdb` file on disk (`assets.tar.gz` is omitted).
  2. On the server, `BINDCRAFTSubmitDesignTool` (`submit_prediction.py`) invokes `resolve_target_structure("4Z18", prefer_format="pdb")` in `foldrun_app/core/target_resolver.py`, which downloads the structure from RCSB PDB (`https://files.rcsb.org/download/4z18.pdb` or `.cif`) and stages it to `gs://<bucket>/targets/<job_name>/`.
  3. Inside the GPU worker, `predict_bindcraft` uses `gemmi` (`st = gemmi.read_structure(...); st.write_pdb(default_target_pdb)`) to guarantee a clean, standard `.pdb` file even if the input was an mmCIF (`.cif`) structure.

---
## 3. Mode A (Interface 1): Drop-In Native CLI (`!bindcraft ...`) with Local `.pdb` Bundling

In the upstream [`martinpacesa/BindCraft`](https://github.com/martinpacesa/BindCraft) repository and `FreeBindCraft` CLI, a design campaign is launched with three JSON configuration files:
1. **`--settings=pdl1_settings.json`** (adapted from [`settings_target/PDL1.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_target/PDL1.json)): Defines `design_path`, `binder_name`, `starting_pdb`, `chains`, `target_hotspot_residues`, `lengths`, and `number_of_final_designs`.
2. **`--filters=default_filters.json`** (from [`settings_filters/default_filters.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_filters/default_filters.json)): Defines acceptance thresholds for `Average_pLDDT`, `Average_pTM`, `Average_i_pTM`, `Average_i_pAE`, `ShapeComplementarity`, `dG`, and `dSASA`.
3. **`--advanced=default_4stage_multimer.json`** (from [`settings_advanced/default_4stage_multimer.json`](https://github.com/martinpacesa/BindCraft/blob/main/settings_advanced/default_4stage_multimer.json)): Configures the 4-stage ColabDesign optimization schedule (`soft_iterations: 75`, `temporary_iterations: 45`, `hard_iterations: 5`, `greedy_iterations: 15`) and SolubleMPNN settings (`enable_mpnn: true`, `mpnn_weights: "soluble"`).

When you run `!bindcraft` (or `!bindcraft.py`) with `foldrun-shim` installed:
- `foldrun_shim.cli.bindcraft_main` intercepts the CLI invocation.
- Host-specific paths (`--settings`, `--filters`, `--advanced`, `--out_dir`) are extracted or stripped on the client (`foldrun-gateway` manages `/opt/bindcraft/settings_filters/default_filters.json` and `/opt/bindcraft/settings_advanced/default_4stage_multimer.json` inside the container).
- The local target structure `./target_pdl1.pdb` referenced in `"starting_pdb": "./target_pdl1.pdb"` is automatically bundled into `assets.tar.gz`.
- Scientific scaling flags (`--no-pyrosetta`, `--num_designs_per_seed=2`, `--num_model_seeds=5`) are validated by `_validate_bindcraft_flags` in `flag_allowlist.py` and dispatched across 5 parallel NVIDIA A100 GPUs.

In [ ]:
import json
import urllib.request
from pathlib import Path

# 1. Ensure local target PDB file (./target_pdl1.pdb) contains full 4Z18 structure
pdb_path = Path("target_pdl1.pdb")
if not pdb_path.exists() or pdb_path.stat().st_size < 10000:
    try:
        urllib.request.urlretrieve("https://files.rcsb.org/download/4Z18.pdb", str(pdb_path))
        print(f"Downloaded 4Z18.pdb to {pdb_path} ({pdb_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"Failed to download 4Z18.pdb ({e}), using inline fallback")
        target_pdl1_pdb = """HEADER    IMMUNE SYSTEM                           06-APR-15   4Z18
TITLE     HUMAN PD-L1 (PROGRAMMED DEATH-LIGAND 1) IGV DOMAIN TARGET
ATOM      1  N   ALA A  18      32.394 -38.055 -22.849  1.00 37.28           N
ATOM      2  CA  ALA A  18      31.602 -39.292 -22.650  1.00 35.59           C
ATOM      3  C   ALA A  18      31.623 -40.168 -23.910  1.00 31.72           C
ATOM      4  O   ALA A  18      31.998 -39.726 -24.996  1.00 32.15           O
ATOM      5  CB  ALA A  18      30.163 -38.945 -22.272  1.00 36.90           C
ATOM      6  N   PHE A  19      31.223 -41.412 -23.739  1.00 26.02           N
ATOM      7  CA  PHE A  19      31.183 -42.395 -24.808  1.00 22.85           C
ATOM      8  C   PHE A  19      30.208 -41.919 -25.888  1.00 22.01           C
ATOM      9  O   PHE A  19      29.042 -41.643 -25.592  1.00 20.89           O
ATOM     10  CB  PHE A  19      30.728 -43.729 -24.197  1.00 22.53           C
ATOM     11  N   ILE A  54      28.177 -54.653 -26.220  1.00 17.55           N
ATOM     12  CA  ILE A  54      26.876 -54.957 -26.764  1.00 17.55           C
ATOM     13  C   ILE A  54      26.633 -54.103 -28.000  1.00 16.09           C
ATOM     14  O   ILE A  54      26.825 -52.894 -27.947  1.00 16.82           O
ATOM     15  CB  ILE A  54      25.769 -54.610 -25.768  1.00 19.79           C
ATOM     16  N   VAL A  55      26.199 -54.738 -29.086  1.00 15.13           N
ATOM     17  CA  VAL A  55      25.818 -54.058 -30.315  1.00 15.27           C
ATOM     18  C   VAL A  55      24.373 -54.374 -30.630  1.00 14.36           C
ATOM     19  O   VAL A  55      24.024 -55.547 -30.721  1.00 14.45           O
ATOM     20  CB  VAL A  55      26.719 -54.474 -31.486  1.00 16.23           C
ATOM     21  N   TYR A  56      23.555 -53.331 -30.811  1.00 13.69           N
ATOM     22  CA  TYR A  56      22.116 -53.468 -30.995  1.00 13.66           C
ATOM     23  C   TYR A  56      21.723 -52.695 -32.246  1.00 13.92           C
ATOM     24  O   TYR A  56      21.989 -51.521 -32.334  1.00 14.70           O
ATOM     25  CB  TYR A  56      21.361 -52.923 -29.765  1.00 13.01           C
ATOM     26  CG  TYR A  56      19.879 -53.248 -29.696  1.00 13.33           C
ATOM     27  CD1 TYR A  56      19.426 -54.342 -28.953  1.00 13.10           C
ATOM     28  CD2 TYR A  56      18.922 -52.487 -30.354  1.00 13.22           C
ATOM     29  CE1 TYR A  56      18.099 -54.653 -28.861  1.00 13.02           C
ATOM     30  CE2 TYR A  56      17.590 -52.789 -30.268  1.00 13.35           C
ATOM     31  CZ  TYR A  56      17.185 -53.873 -29.520  1.00 13.18           C
ATOM     32  OH  TYR A  56      15.862 -54.179 -29.431  1.00 13.84           O
TER      33      TYR A  56
END
"""
        pdb_path.write_text(target_pdl1_pdb, encoding="utf-8")
else:
    print(f"Using existing {pdb_path} ({pdb_path.stat().st_size} bytes)")

# 2. Write pdl1_settings.json (adapted from martinpacesa/BindCraft/settings_target/PDL1.json)
pdl1_settings = {
    "design_path": "./bindcraft_out/PDL1/",
    "binder_name": "PDL1",
    "starting_pdb": "./target_pdl1.pdb",
    "chains": "A",
    "target_hotspot_residues": "56",
    "lengths": [65, 150],
    "number_of_final_designs": 2,
}
Path("pdl1_settings.json").write_text(json.dumps(pdl1_settings, indent=2), encoding="utf-8")

# 3. Write local custom copies of default_filters.json and default_4stage_multimer.json:
#    foldrun-shim bundles both --filters and --advanced JSONs into assets.tar.gz alongside starting_pdb!
Path("default_filters.json").write_text(
    json.dumps(
        {
            "Average_pLDDT": {"threshold": 0.8, "higher": True},
            "Average_i_pTM": {"threshold": 0.5, "higher": True},
        },
        indent=2,
    ),
    encoding="utf-8",
)
Path("default_4stage_multimer.json").write_text(
    json.dumps(
        {
            "design_algorithm": "4stage",
            "use_multimer_design": True,
            "soft_iterations": 75,
            "temporary_iterations": 45,
            "hard_iterations": 5,
            "greedy_iterations": 15,
            "enable_mpnn": True,
            "mpnn_weights": "soluble",
        },
        indent=2,
    ),
    encoding="utf-8",
)

# 4. Inspect how foldrun_shim.bundler collects ./target_pdl1.pdb, --filters, and --advanced into assets.tar.gz
bundled_local = bundle_local_inputs(
    model="bindcraft",
    input_path="pdl1_settings.json",
    cli_args=[
        "--filters",
        "default_filters.json",
        "--advanced",
        "default_4stage_multimer.json",
        "--no-pyrosetta",
        "--num_designs_per_seed=2",
        "--num_model_seeds=5",
    ],
)
est_residues = 250 + max(pdl1_settings["lengths"])
gpu_tier = "A100_80GB (a2-ultragpu-1g)" if est_residues > 450 else "A100 40GB (a2-highgpu-1g)"

print(f"Primary settings file           : {bundled_local.input_filename}")
print(f"Bundled assets in assets.tar.gz : {list(bundled_local.assets_map.keys())}")
print(f"Archive size (compressed bytes) : {len(bundled_local.assets_tar_bytes or b'')}")
print(f"Deterministic client_input_sha256: {bundled_local.client_input_sha256}")
print(
    f"Estimated complex residues      : ~{est_residues} residues -> Auto-selected GPU: {gpu_tier}"
)

In [ ]:
# Interface 1: Drop-In Native BindCraft CLI (FreeBindCraft --no-pyrosetta)
# foldrun-shim intercepts `bindcraft` (and `bindcraft.py`), bundles ./target_pdl1.pdb plus custom
# --filters and --advanced JSONs into assets.tar.gz, and fans out 5 seeds across Vertex AI A100 GPUs.
!bindcraft --settings=pdl1_settings.json --filters=default_filters.json --advanced=default_4stage_multimer.json --no-pyrosetta --out_dir=./bindcraft_out --num_designs_per_seed=2 --num_model_seeds=5

---
## 4. Mode A (Interface 2): Python Notebook Helper API (`bindcraft.predict`) & `.foldrun_job.json` Recovery

In addition to the `!bindcraft` / `!bindcraft.py` CLI entrypoints, `foldrun-shim` exports a Pythonic notebook helper (`from foldrun_shim import bindcraft`) that:
1. Accepts allowlisted BindCraft scientific arguments (`_validate_bindcraft_flags` in `flag_allowlist.py`) as Python keyword arguments:
   - `num_designs_per_seed` / `number_of_final_designs` (`1..20`, default `2`): Final accepted binder designs generated per seed trajectory.
   - `num_model_seeds` / `num_seeds` (`1..50`, default `5`): Number of parallel Vertex AI A100 GPU trajectories (`dsl.ParallelFor`), producing `num_model_seeds * num_designs_per_seed` total candidate designs.
   - `filters` / `advanced` (`str | Path`, optional): Custom filter thresholds or advanced stage settings JSON files, automatically bundled into `assets.tar.gz` alongside `starting_pdb`.
   - `seed` (`int`, optional): Deterministic base random seed passed to `ConfigureSeeds` (`base_seed`).
   - `binder_length_range` (`str`, default `"80..140"`): Optional binder length range override (e.g., `"65..150"`).
   - `target_chains` (`str`, optional): Target chains override (e.g., `"A"`).
   - `binding_residues` (`str`, optional): Target hotspot residues override (e.g., `"54,56,66"`).
2. Automatically reads `"design_path"` from your `--settings` JSON when `out_dir` is omitted.
3. Returns a `GatewayJobResult` object that renders an interactive HTML status card in Jupyter (`job._repr_html_()`) with a direct link to the **3D FoldRun Viewer** (`job.viewer_url`).
4. Persists `<out_dir>/.foldrun_job.json` so that if your browser tab closes or Colab disconnects during multi-stage AF2 backpropagation, re-running the cell automatically re-attaches to the in-flight Vertex AI `PipelineJob`.

In [ ]:
# Interface 2: Python Notebook Helper API (5 parallel A100 seeds x 2 designs/seed = 10 candidate binders)
job = bindcraft.predict(
    "pdl1_settings.json",
    out_dir="./bindcraft_out",
    num_designs_per_seed=2,
    num_model_seeds=5,
    seed=42,
)

# Renders interactive GatewayJobResult HTML card in Jupyter / Colab with job.viewer_url
job

In [ ]:
# Inspect <out_dir>/.foldrun_job.json written by foldrun-shim for disconnect resilience
state_file = Path("./bindcraft_out/.foldrun_job.json")
if state_file.exists():
    state_data = json.loads(state_file.read_text(encoding="utf-8"))
    print("Saved disconnect-recovery state (.foldrun_job.json):")
    print(json.dumps(state_data, indent=2))
else:
    print(
        "No .foldrun_job.json found yet (run the prediction cell above against a live foldrun-gateway)."
    )

---
## 5. Mode B: Server-Side RCSB PDB Resolution (`"starting_pdb": "4Z18"`) Without Local Structure Files

In standard BindCraft, `starting_pdb` must point to a local `.pdb` file on disk (`/content/bindcraft/example/PDL1.pdb`). With FoldRun, you can also pass a **4-character RCSB PDB ID** directly in `"starting_pdb": "4Z18"`:
1. `foldrun_shim.bundler._collect_bindcraft_assets` recognizes `"4Z18"` as a 4-character PDB identifier (`^[0-9][A-Za-z0-9]{3}$`) rather than a local file path, so `assets_map` is empty (`[]`) and no local `.pdb` file is needed.
2. On `foldrun-gateway`, `BINDCRAFTSubmitDesignTool` (`submit_prediction.py`) detects the 4-character PDB ID and calls `resolve_target_structure("4Z18", prefer_format="pdb")` in `foldrun_app/core/target_resolver.py`, which downloads `4Z18` directly from RCSB PDB and uploads it to `gs://<bucket>/targets/<job_name>/4z18.pdb` (or `.cif`).
3. Inside the A100 GPU container, `predict_bindcraft` (`predict.py`) reads the staged target structure with `gemmi` (`st = gemmi.read_structure(target_structure.path); st.write_pdb(default_target_pdb)`) so `FreeBindCraft` always receives a sanitized `.pdb` file.

In [ ]:
# Write a BindCraft target settings JSON that references RCSB PDB ID "4Z18" directly in starting_pdb
pdl1_rcsb_settings = {
    "design_path": "./bindcraft_rcsb_out/PDL1_4Z18/",
    "binder_name": "PDL1_4Z18",
    "starting_pdb": "4Z18",
    "chains": "A",
    "target_hotspot_residues": "54,56,66",
    "lengths": [65, 120],
    "number_of_final_designs": 2,
}
Path("pdl1_rcsb_4z18_settings.json").write_text(
    json.dumps(pdl1_rcsb_settings, indent=2),
    encoding="utf-8",
)

# Verify that foldrun_shim.bundler does NOT require a local file when starting_pdb is a 4-char PDB ID
bundled_rcsb = bundle_local_inputs(
    model="bindcraft",
    input_path="pdl1_rcsb_4z18_settings.json",
    cli_args=["--no-pyrosetta", "--num_designs_per_seed=2", "--num_model_seeds=5"],
)
print(f"Bundled assets for RCSB ID '4Z18': {list(bundled_rcsb.assets_map.keys())} (expected: [])")
print(f"Deterministic client_input_sha256 : {bundled_rcsb.client_input_sha256}")

# Submit Mode B BindCraft campaign using server-side RCSB PDB resolution + shared NFS weights
rcsb_job = bindcraft.predict(
    "pdl1_rcsb_4z18_settings.json",
    out_dir="./bindcraft_rcsb_out",
    num_designs_per_seed=2,
    num_model_seeds=5,
    seed=42,
)
rcsb_job

---
## 6. Downstream Visualization & Biophysical Metrics Analysis

When the parallel Vertex AI A100 pipeline finishes, `foldrun-gateway` (`_package_generative_binder_outputs` in `output_packager.py`) collects all per-seed structures and confidence records from GCS, ranks them globally by `ranking_score` descending, and extracts them into `--out_dir` under `<out_dir>/<stem>/`:

```text
bindcraft_out/
├── .foldrun_job.json
└── pdl1_settings/
    ├── Accepted/
    │   ├── final_design_stats.csv              # Global ranked table compatible with BindCraft.ipynb
    │   ├── ranked_0.cif                        # Top-ranked binder + target complex structure
    │   └── ranked_1.cif
    ├── ranked_designs/
    │   ├── metrics.csv                         # Unified FoldRun generative binder metrics table
    │   ├── ranked_0.cif
    │   └── ranked_1.cif
    └── predictions/
        ├── design_seed<seed>_0.cif             # Per-seed normalized mmCIF structures
        ├── confidence_design_seed<seed>_0.json # Per-seed JSON metrics (plddt, iptm, ptm, dG, dSASA)
        └── ...
```

Below we:
1. Load `<out_dir>/<stem>/Accepted/final_design_stats.csv` and `<out_dir>/<stem>/predictions/confidence_design_seed*_*.json`.
2. Plot `ranking_score`, `plddt`, `iptm`, `interface_energy` ($\Delta G$, kcal/mol from OpenMM/FreeBindCraft interface scoring), and `contact_surface_area` ($\Delta\text{SASA}$, $\text{\AA}^2$ from FreeSASA) across candidate binders with `matplotlib`.
3. Render `<out_dir>/<stem>/Accepted/ranked_0.cif` (and `<out_dir>/<stem>/predictions/design_seed*_*.cif`) in `py3Dmol` with the target chain (`A`), hotspot residue (`Tyr56`), and *de novo* designed binder (`B`) highlighted.

In [ ]:
# 6.1 Load materialized BindCraft outputs from <out_dir>/<stem>/Accepted/final_design_stats.csv
#     and <out_dir>/<stem>/predictions/confidence_design_seed*_*.json
import os

import pandas as pd

candidate_stem_dirs = [
    Path("./bindcraft_out/pdl1_settings"),
    Path("./results/pdl1_binder/PDL1"),
    Path("./results/pdl1_binder"),
]
stem_dir = next((p for p in candidate_stem_dirs if p.exists()), candidate_stem_dirs[0])
accepted_csv = stem_dir / "Accepted" / "final_design_stats.csv"
ranked_csv = stem_dir / "ranked_designs" / "metrics.csv"
predictions_dir = stem_dir / "predictions"

synthetic_demo_mode = os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true"
records = []
if accepted_csv.exists():
    df_stats = pd.read_csv(accepted_csv)
    records = df_stats.to_dict(orient="records")
    print(f"Loaded {len(records)} ranked BindCraft designs from {accepted_csv}:\n")
elif ranked_csv.exists():
    df_stats = pd.read_csv(ranked_csv)
    records = df_stats.to_dict(orient="records")
    print(f"Loaded {len(records)} ranked BindCraft designs from {ranked_csv}:\n")
elif predictions_dir.exists() and list(predictions_dir.glob("confidence_design_seed*.json")):
    for jf in sorted(predictions_dir.glob("confidence_design_seed*.json")):
        records.append(json.loads(jf.read_text(encoding="utf-8")))
    records.sort(key=lambda r: float(r.get("ranking_score", 0.0)), reverse=True)
    df_stats = pd.DataFrame(records)
    print(f"Loaded {len(records)} per-seed confidence JSONs from {predictions_dir}:\n")
elif synthetic_demo_mode:
    print("=" * 80)
    print("SYNTHETIC DEMO DATA (NOT A BINDCRAFT PREDICTION) — FOLDRUN_SYNTHETIC_DEMO=true")
    print("=" * 80)
    records = [
        {
            "rank": 0,
            "design_id": "design_seed184920_0",
            "seed_value": 184920,
            "ranking_score": 0.885,
            "plddt": 89.4,
            "iptm": 0.876,
            "ptm": 0.882,
            "interface_energy": -46.8,
            "contact_surface_area": 1540.0,
            "designed_sequence": "MDEELRKALEELKKLGEELKKLGPELAEALKELAEKLGEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGE",
        },
        {
            "rank": 1,
            "design_id": "design_seed739104_0",
            "seed_value": 739104,
            "ranking_score": 0.869,
            "plddt": 87.8,
            "iptm": 0.860,
            "ptm": 0.865,
            "interface_energy": -43.2,
            "contact_surface_area": 1465.0,
            "designed_sequence": "MSEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGEEVKKLLEELKKLGEELKKLGPELAEALKELAEKLGE",
        },
        {
            "rank": 2,
            "design_id": "design_seed502193_1",
            "seed_value": 502193,
            "ranking_score": 0.854,
            "plddt": 86.2,
            "iptm": 0.846,
            "ptm": 0.851,
            "interface_energy": -40.5,
            "contact_surface_area": 1390.0,
            "designed_sequence": "MKEALEELKKLGEELKKLGPELAEALKELAEKLGEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGEEVKK",
        },
        {
            "rank": 3,
            "design_id": "design_seed184920_1",
            "seed_value": 184920,
            "ranking_score": 0.841,
            "plddt": 85.0,
            "iptm": 0.832,
            "ptm": 0.840,
            "interface_energy": -38.1,
            "contact_surface_area": 1325.0,
            "designed_sequence": "MEEALKKLAEKLGEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGEELKKLGPELAEALKELAEKLGEEVK",
        },
        {
            "rank": 4,
            "design_id": "design_seed918273_0",
            "seed_value": 918273,
            "ranking_score": 0.826,
            "plddt": 83.6,
            "iptm": 0.816,
            "ptm": 0.828,
            "interface_energy": -35.9,
            "contact_surface_area": 1260.0,
            "designed_sequence": "MPEELAEALKELAEKLGEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGEELKKLGEELKKLGPELAEALK",
        },
        {
            "rank": 5,
            "design_id": "design_seed394812_0",
            "seed_value": 394812,
            "ranking_score": 0.810,
            "plddt": 82.1,
            "iptm": 0.799,
            "ptm": 0.812,
            "interface_energy": -33.4,
            "contact_surface_area": 1195.0,
            "designed_sequence": "MAEELKKLGEELKKLGPELAEALKELAEKLGEEVKKLLEEAKEAGVDPAAYKEALKKLAEKLGEEVKKLLE",
        },
    ]
    df_stats = pd.DataFrame(records)
else:
    found_files = (
        sorted(
            p.relative_to(stem_dir).as_posix()
            for p in stem_dir.rglob("*")
            if p.is_file() and p.name != ".foldrun_job.json"
        )
        if stem_dir.exists()
        else []
    )
    raise FileNotFoundError(
        f"No BindCraft design outputs found in {stem_dir.resolve()}.\n"
        f"Files currently in {stem_dir}: {found_files or ['<empty>']}\n"
        f"Run a BindCraft prediction cell above against a live FoldRun Gateway first "
        f"(or set FOLDRUN_SYNTHETIC_DEMO=true strictly to test notebook chart layout offline)."
    )

print(
    f"{'Rank':<5} {'Design ID':<22} {'Seed':>10} {'Rank Score':>11} {'pLDDT':>8} {'ipTM':>8} {'dG (kcal/mol)':>15} {'dSASA (A^2)':>13}"
)
print("-" * 98)
for idx, r in enumerate(records):
    print(
        f"{idx:<5} "
        f"{r.get('design_id', f'design_{idx}')!s:<22} "
        f"{int(r.get('seed_value', 0)):>10} "
        f"{float(r.get('ranking_score', 0.0)):>11.3f} "
        f"{float(r.get('plddt', 0.0)):>8.1f} "
        f"{float(r.get('iptm', 0.0)):>8.3f} "
        f"{float(r.get('interface_energy', 0.0)):>15.1f} "
        f"{float(r.get('contact_surface_area', 0.0)):>13.1f}"
    )

In [ ]:
# 6.2 Plot ranking_score, plddt, iptm, interface_energy (dG), and contact_surface_area (dSASA) with matplotlib
import matplotlib.pyplot as plt
import numpy as np

labels = [
    str(r.get("design_id", f"design_{i}")).replace("design_", "") for i, r in enumerate(records)
]
ranking_scores = [float(r.get("ranking_score", 0.0)) for r in records]
plddts = [float(r.get("plddt", 0.0)) for r in records]
iptms = [float(r.get("iptm", 0.0)) for r in records]
dg_vals = [float(r.get("interface_energy", -35.0)) for r in records]
dsasa_vals = [float(r.get("contact_surface_area", 1250.0)) for r in records]

x = np.arange(len(labels))
width = 0.26

fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
if os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true":
    fig.suptitle(
        "SYNTHETIC DEMO DATA (NOT A BINDCRAFT PREDICTION)",
        color="#b31412",
        fontweight="bold",
    )

# Panel 1: Ranking Score, ipTM, and pLDDT / 100
ax0 = axes[0, 0]
ax0.bar(x - width, ranking_scores, width, label="Ranking Score", color="#1a73e8")
ax0.bar(x, iptms, width, label="Interface pTM (ipTM)", color="#34a853")
ax0.bar(x + width, [p / 100.0 for p in plddts], width, label="pLDDT / 100", color="#fbbc04")
ax0.axhline(
    0.80,
    color="#5f6368",
    linestyle="--",
    linewidth=1.0,
    label="Default Filter Threshold (0.80)",
)
ax0.set_title("AF2 Multimer Confidence & Ranking Score Across Seeds")
ax0.set_xticks(x)
ax0.set_xticklabels(labels, rotation=25, ha="right")
ax0.set_ylim(0.0, 1.05)
ax0.set_ylabel("Normalized Score (0 - 1)")
ax0.legend(loc="lower right", fontsize=8.5)
ax0.grid(axis="y", alpha=0.25)

# Panel 2: Biophysical Interface Energy (dG) vs Contact Surface Area (dSASA)
ax1 = axes[0, 1]
sc = ax1.scatter(
    dsasa_vals,
    dg_vals,
    c=ranking_scores,
    cmap="viridis",
    s=110,
    edgecolors="black",
    zorder=3,
)
for i, lbl in enumerate(labels):
    ax1.annotate(
        lbl,
        (dsasa_vals[i], dg_vals[i]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=8.5,
    )
ax1.set_title("Biophysical Interface Energy (dG) vs. Contact Surface Area (dSASA)")
ax1.set_xlabel("Contact Surface Area dSASA (Å²)")
ax1.set_ylabel("Interface Free Energy dG (kcal/mol, more negative = tighter)")
ax1.grid(True, alpha=0.25)
plt.colorbar(sc, ax=ax1, label="Ranking Score")

# Panel 3: Per-Design Interface Energy (dG) + Mean pLDDT overlay
ax2 = axes[1, 0]
bars_dg = ax2.bar(x, dg_vals, width=0.5, color="#B76E79", edgecolor="#3c5b6f")
ax2.set_title("OpenMM / FreeBindCraft Interface Energy (dG)")
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=25, ha="right")
ax2.set_ylabel("Interface Energy dG (kcal/mol)")
ax2.grid(axis="y", alpha=0.25)

# Panel 4: Per-Design Contact Surface Area (dSASA) from FreeSASA
ax3 = axes[1, 1]
ax3.bar(x, dsasa_vals, width=0.5, color="#3c5b6f", edgecolor="#1a73e8")
ax3.set_title("FreeSASA Buried Contact Surface Area (dSASA)")
ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=25, ha="right")
ax3.set_ylabel("Buried Interface Area dSASA (Å²)")
ax3.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Interactive 3D Visualization of Designed Binder + Target Complex in py3Dmol
#     Loads <out_dir>/<stem>/Accepted/ranked_0.cif (or <out_dir>/<stem>/predictions/design_seed*_*.cif)
import os

import py3Dmol

accepted_ranked_0 = stem_dir / "Accepted" / "ranked_0.cif"
ranked_designs_0 = stem_dir / "ranked_designs" / "ranked_0.cif"
pred_cifs = (
    sorted((stem_dir / "predictions").glob("design_seed*.cif"))
    if (stem_dir / "predictions").exists()
    else []
)

structure_path = None
structure_format = "cif"
if accepted_ranked_0.exists():
    structure_path = accepted_ranked_0
elif ranked_designs_0.exists():
    structure_path = ranked_designs_0
elif pred_cifs:
    structure_path = pred_cifs[0]

if structure_path is not None:
    structure_text = structure_path.read_text(encoding="utf-8")
    print(f"Rendering materialized BindCraft structure: {structure_path}")
elif os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true":
    structure_format = "pdb"
    structure_text = (
        Path("target_pdl1.pdb").read_text(encoding="utf-8").replace("END\n", "")
        + """ATOM     34  N   GLU B   1      20.110 -50.210 -27.420  1.00 91.20           N
ATOM     35  CA  GLU B   1      19.240 -51.180 -26.780  1.00 91.20           C
ATOM     36  C   GLU B   1      18.120 -50.490 -26.010  1.00 91.20           C
ATOM     37  O   GLU B   1      17.890 -49.300 -26.190  1.00 91.20           O
ATOM     38  N   LEU B   2      17.430 -51.250 -25.160  1.00 92.50           N
ATOM     39  CA  LEU B   2      16.320 -50.720 -24.380  1.00 92.50           C
ATOM     40  C   LEU B   2      16.810 -49.680 -23.380  1.00 92.50           C
ATOM     41  O   LEU B   2      16.190 -48.630 -23.220  1.00 92.50           O
ATOM     42  N   LYS B   3      17.940 -49.990 -22.740  1.00 93.10           N
ATOM     43  CA  LYS B   3      18.540 -49.090 -21.760  1.00 93.10           C
ATOM     44  C   LYS B   3      18.820 -47.720 -22.360  1.00 93.10           C
ATOM     45  O   LYS B   3      18.390 -46.710 -21.810  1.00 93.10           O
END
"""
    )
    print("SYNTHETIC DEMO DATA (NOT A BINDCRAFT PREDICTION) — rendering synthetic preview complex.")
else:
    raise FileNotFoundError(
        f"No BindCraft 3D structure files (*.cif) found under {stem_dir.resolve()}. "
        f"Run a BindCraft prediction cell above against a live FoldRun Gateway first."
    )

view = py3Dmol.view(width=860, height=520)
view.addModel(structure_text, structure_format)
view.setBackgroundColor("white")

# Style Target Chain A (#3c5b6f) and Designed Binder Chain B (#B76E79) matching upstream BindCraft.ipynb
view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "#B76E79"}})

# Highlight target hotspot residue Tyr56 (and neighboring 54, 55) on Chain A
view.addStyle(
    {"chain": "A", "resi": [54, 55, 56]},
    {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}},
)
view.addSurface(
    py3Dmol.VDW,
    {"opacity": 0.35, "color": "#fbbc04"},
    {"chain": "A", "resi": [56]},
)

view.zoomTo()
view.show()

if "job" in globals() and getattr(job, "viewer_url", None):
    print(f"Full interactive FoldRun 3D Viewer URL: {job.viewer_url}")